# Coringa Mangrove Classification — Landsat-8 2017

Comparing RF, XGBoost, and DeepLabV3+ on Landsat-8 OLI data.

**Config:**
- Sensor: Landsat-8 OLI (30m resolution)
- Date: Sep–Dec 2017
- Patch size: 64×64 (1920m × 1920m per patch)
- Features: 6 Landsat bands + 5 spectral indices = 11 channels
- Labels: ESA WorldCover 2021 (class 95 = mangroves)

In [12]:
!pip install earthengine-api geemap scikit-learn xgboost segmentation_models_pytorch -q
!earthengine authenticate

I0000 00:00:1793642235.287977    3107 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793642237.948759    3107 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Authenticate: Limited support in Colab. Use ee.Authenticate() or --auth_mode=notebook instead.
Authenticate: Credentials already exist. Use --force to refresh.


In [13]:
import ee
import geemap
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

ee.Initialize(project='coringa-mangrove')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## 1. Constants

In [14]:
CORINGA_BBOX = [82.13, 16.42, 82.23, 17.0]
REGION = ee.Geometry.Rectangle(CORINGA_BBOX)

# Landsat-8 date range (Sep-Dec 2017, matching paper)
START_DATE = '2017-09-01'
END_DATE = '2017-12-31'

# Landsat-8 band names
L8_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7']  # Blue, Green, Red, NIR, SWIR1, SWIR2

# Patch config (30m resolution, use smaller patches)
PATCH_SIZE = 64

# Training config
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4

# Spectral indices
INDEX_BANDS = ['NDVI', 'NDMI', 'MNDWI', 'GCVI', 'SR']
ALL_FEATURES = L8_BANDS + INDEX_BANDS
N_CHANNELS = len(ALL_FEATURES)  # 11

MANGROVE_CLASS = 95

## 2. Load Landsat-8 & Compute Indices

In [15]:
def load_landsat8():
    """Load and composite Landsat-8 OLI for the study period."""
    # Landsat-8 Collection 2 Level 2 SR band names are prefixed with 'SR_'
    sr_bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']

    l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
          .filterBounds(REGION)
          .filterDate(START_DATE, END_DATE)
          .filter(ee.Filter.lt('CLOUD_COVER', 20))
          .median()
          .clip(REGION))

    # Select the SR bands
    selected = l8.select(sr_bands)

    # Apply scaling factors (Collection 2 Level 2 SR: scale = 0.0000275, offset = -0.2)
    scaled = selected.multiply(0.0000275).add(-0.2)

    # Rename bands to standard names ['B2', 'B3', 'B4', 'B5', 'B6', 'B7'] for the pipeline
    return scaled.rename(L8_BANDS)

def compute_indices(image):
    """Compute spectral indices from Landsat-8 bands."""
    # Landsat-8: B5=NIR, B4=Red, B3=Green, B6=SWIR1, B7=SWIR2
    ndvi = image.normalizedDifference(['B5', 'B4']).rename('NDVI')
    ndmi = image.normalizedDifference(['B5', 'B7']).rename('NDMI')
    mndwi = image.normalizedDifference(['B3', 'B6']).rename('MNDWI')
    gcvi = image.expression('B5 / B3 - 1', {
        'B5': image.select('B5'), 'B3': image.select('B3')
    }).rename('GCVI')
    sr = image.expression('B5 / B4', {
        'B5': image.select('B5'), 'B4': image.select('B4')
    }).rename('SR')
    return image.addBands([ndvi, ndmi, mndwi, gcvi, sr])

def prepare_features():
    """Full pipeline: load L8, compute indices, return stacked image."""
    l8 = load_landsat8()
    with_indices = compute_indices(l8)
    return with_indices.select(ALL_FEATURES)

print('Loading Landsat-8 2017 imagery...')
features_image = prepare_features()
print('Feature bands:', features_image.bandNames().getInfo())

Loading Landsat-8 2017 imagery...
Feature bands: ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'NDVI', 'NDMI', 'MNDWI', 'GCVI', 'SR']


## 3. Load WorldCover Labels

In [16]:
def load_worldcover():
    """Load ESA WorldCover 2021 and create binary mangrove mask."""
    wc = ee.Image('ESA/WorldCover/v200/2021').clip(REGION)
    # Create an anonymous constant image and use .where() to strip the lineage metadata
    is_mangrove = wc.eq(MANGROVE_CLASS)
    mangrove_mask = ee.Image.constant(0).where(is_mangrove, 1).rename('mangrove').toByte()
    return mangrove_mask

print('Loading WorldCover labels...')
labels_image = load_worldcover()
print('Labels loaded.')

Loading WorldCover labels...
Labels loaded.


## 4. Download to Numpy & Extract Patches

In [18]:
# Combine features + labels into one image
combined = features_image.addBands(labels_image)

print('Downloading combined image (Landsat-8 100m)...')
# Increase scale to 100m to keep request size below Earth Engine API limits
combined_array = geemap.ee_to_numpy(combined, region=REGION, scale=100)
print(f'Combined shape: {combined_array.shape}')

# Split features and labels
features_array = combined_array[:, :, :N_CHANNELS]
labels_array = combined_array[:, :, N_CHANNELS:]

print(f'Features: {features_array.shape}, Labels: {labels_array.shape}')
print(f'Mangrove ratio: {(labels_array == 1).mean():.4f}')

Combined shape: (647, 113, 12)
Features: (647, 113, 11), Labels: (647, 113, 1)
Mangrove ratio: 0.0165


In [19]:
# Extract patches from numpy array
def extract_patches(features, labels, patch_size=64, n_patches=1000):
    h, w, c = features.shape
    X_list = []
    y_list = []

    np.random.seed(42)
    for i in range(n_patches):
        r = np.random.randint(0, h - patch_size)
        c_idx = np.random.randint(0, w - patch_size)

        patch_x = features[r:r+patch_size, c_idx:c_idx+patch_size]
        patch_y = labels[r:r+patch_size, c_idx:c_idx+patch_size]

        X_list.append(patch_x.transpose(2, 0, 1))  # (C, H, W)
        y_list.append(patch_y.transpose(2, 0, 1))  # (1, H, W)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    # Normalize per channel
    for c in range(X.shape[1]):
        mean = X[:, c].mean()
        std = X[:, c].std() + 1e-8
        X[:, c] = (X[:, c] - mean) / std

    return X, y

X, y = extract_patches(features_array, labels_array, PATCH_SIZE, n_patches=1000)
print(f'X: {X.shape}, y: {y.shape}')
print(f'Mangrove ratio: {y.mean():.4f}')

X: (1000, 11, 64, 64), y: (1000, 1, 64, 64)
Mangrove ratio: 0.0088


## 5. Dataset & DataLoader

In [20]:
class MangroveDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Split: 70/15/15
n = len(X)
idx = np.random.permutation(n)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

train_loader = DataLoader(MangroveDataset(X[idx[:n_train]], y[idx[:n_train]]),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(MangroveDataset(X[idx[n_train:n_train+n_val]], y[idx[n_train:n_train+n_val]]),
                        batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(MangroveDataset(X[idx[n_train+n_val:]], y[idx[n_train+n_val:]]),
                         batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {n_train}, Val: {n_val}, Test: {n - n_train - n_val}')

Train: 700, Val: 150, Test: 150


## 6. RF & XGBoost (pixel-level)

In [21]:
# Flatten patches to pixels
X_pixels = X.reshape(X.shape[0], X.shape[1], -1).transpose(0, 2, 1).reshape(-1, N_CHANNELS)
y_pixels = y.reshape(-1)

# Subsample 100k pixels for speed
sub_idx = np.random.choice(len(X_pixels), 100000, replace=False)
X_sub = X_pixels[sub_idx]
y_sub = y_pixels[sub_idx]

X_train_mL, X_test_mL, y_train_mL, y_test_mL = train_test_split(
    X_sub, y_sub, test_size=0.3, random_state=42, stratify=y_sub
)
print(f'Train: {len(X_train_mL)}, Test: {len(X_test_mL)}')

Train: 70000, Test: 30000


In [22]:
# RF
rf = RandomForestClassifier(n_estimators=100, max_features=5, random_state=42, n_jobs=-1)
rf.fit(X_train_mL, y_train_mL)
y_pred_rf = rf.predict(X_test_mL)

def compute_iou(y_true, y_pred):
    intersection = (y_true.astype(bool) & y_pred.astype(bool)).sum()
    union = (y_true.astype(bool) | y_pred.astype(bool)).sum()
    return intersection / (union + 1e-7)

print(f'RF Accuracy: {accuracy_score(y_test_mL, y_pred_rf):.4f}')
print(f'RF IoU: {compute_iou(y_test_mL, y_pred_rf):.4f}')

RF Accuracy: 0.9983
RF IoU: 0.8083


In [23]:
# XGBoost
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_train_mL, y_train_mL)
y_pred_xgb = xgb.predict(X_test_mL)

print(f'XGBoost Accuracy: {accuracy_score(y_test_mL, y_pred_xgb):.4f}')
print(f'XGBoost IoU: {compute_iou(y_test_mL, y_pred_xgb):.4f}')

XGBoost Accuracy: 0.9979
XGBoost IoU: 0.7701


## 7. DeepLabV3+ Training

In [24]:
def compute_iou_dl(preds, targets, threshold=0.5):
    preds = (preds > threshold).float()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    return ((intersection + 1e-7) / (union + 1e-7)).item()

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    total_iou = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch)
            total_loss += criterion(preds, y_batch).item()
            total_iou += compute_iou_dl(preds, y_batch)
    return total_loss / len(loader), total_iou / len(loader)

In [25]:
model = smp.DeepLabV3Plus(
    encoder_name='efficientnet-b0',
    encoder_weights='imagenet',
    in_channels=N_CHANNELS,
    classes=1,
).to(device)

criterion = smp.losses.FocalLoss(mode='binary', alpha=0.75, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_iou = 0

print('Training DeepLabV3+...')
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_iou = validate(model, val_loader, criterion)

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), 'best_model_l8.pt')

    print(f'Epoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | IoU: {val_iou:.4f}')

print(f'\nBest Val IoU: {best_val_iou:.4f}')

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Training DeepLabV3+...
Epoch 1/15 | Train: 0.0280 | Val: 0.0249 | IoU: 0.1000
Epoch 2/15 | Train: 0.0136 | Val: 0.0116 | IoU: 0.1000
Epoch 3/15 | Train: 0.0082 | Val: 0.0070 | IoU: 0.1557
Epoch 4/15 | Train: 0.0059 | Val: 0.0054 | IoU: 0.3723
Epoch 5/15 | Train: 0.0043 | Val: 0.0041 | IoU: 0.4189
Epoch 6/15 | Train: 0.0034 | Val: 0.0032 | IoU: 0.5307
Epoch 7/15 | Train: 0.0029 | Val: 0.0030 | IoU: 0.5421
Epoch 8/15 | Train: 0.0024 | Val: 0.0025 | IoU: 0.5940
Epoch 9/15 | Train: 0.0021 | Val: 0.0020 | IoU: 0.6026
Epoch 10/15 | Train: 0.0019 | Val: 0.0020 | IoU: 0.6540
Epoch 11/15 | Train: 0.0017 | Val: 0.0017 | IoU: 0.6748
Epoch 12/15 | Train: 0.0016 | Val: 0.0016 | IoU: 0.6739
Epoch 13/15 | Train: 0.0014 | Val: 0.0016 | IoU: 0.6713
Epoch 14/15 | Train: 0.0015 | Val: 0.0014 | IoU: 0.6723
Epoch 15/15 | Train: 0.0013 | Val: 0.0012 | IoU: 0.7005

Best Val IoU: 0.7005


## 8. DL Test Evaluation

In [26]:
model.load_state_dict(torch.load('best_model_l8.pt'))
model.eval()

all_preds = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)
        preds_binary = (preds > 0.5).float().cpu().numpy()
        all_preds.append(preds_binary)
        all_labels.append(y_batch.numpy())

dl_preds = np.concatenate(all_preds).flatten()
dl_labels = np.concatenate(all_labels).flatten()

dl_acc = (dl_preds == dl_labels).mean()
dl_iou = compute_iou(dl_labels, dl_preds)

print(f'DeepLabV3+ Accuracy: {dl_acc:.4f}')
print(f'DeepLabV3+ IoU: {dl_iou:.4f}')

DeepLabV3+ Accuracy: 0.9948
DeepLabV3+ IoU: 0.6950


## 9. Final Comparison

In [27]:
print('=' * 50)
print('LANDSAT-8 2017 — CORINGA MANGROVE CLASSIFICATION')
print('=' * 50)
print(f'RF+CT       | Accuracy: {accuracy_score(y_test_mL, y_pred_rf):.4f} | IoU: {compute_iou(y_test_mL, y_pred_rf):.4f}')
print(f'XGBoost     | Accuracy: {accuracy_score(y_test_mL, y_pred_xgb):.4f} | IoU: {compute_iou(y_test_mL, y_pred_xgb):.4f}')
print(f'DeepLabV3+  | Accuracy: {dl_acc:.4f} | IoU: {dl_iou:.4f}')
print('=' * 50)

LANDSAT-8 2017 — CORINGA MANGROVE CLASSIFICATION
RF+CT       | Accuracy: 0.9983 | IoU: 0.8083
XGBoost     | Accuracy: 0.9979 | IoU: 0.7701
DeepLabV3+  | Accuracy: 0.9948 | IoU: 0.6950
